<a href="https://colab.research.google.com/github/Udaythecoder/BLUESTOCK/blob/main/mutual_funds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
#Day-1


In [1]:
!pip install scipy

In [2]:
!pip install sqlalchemy

In [3]:
!pip install requests

In [5]:
!pip install jupyter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.5/12.5 MB 93.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 110.9 MB/s eta 0:00:00


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
d1=pd.read_csv('/01_fund_master.csv')
print()

In [3]:
print(d1.shape)

(40, 15)


In [4]:
print(d1.dtypes)

amfi_code               int64
fund_house             object
scheme_name            object
category               object
sub_category           object
plan                   object
launch_date            object
benchmark              object
expense_ratio_pct     float64
exit_load_pct         float64
min_sip_amount          int64
min_lumpsum_amount      int64
fund_manager           object
risk_category          object
sebi_category_code     object
dtype: object


In [5]:
display(d1.head())

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [9]:
d2 = pd.read_csv('/02_nav_history.csv')
display(d2.head())

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [10]:
key_schemes_amfi_codes = [119551, 120503, 118632, 119092, 120841]

# Filter d1 to get scheme names for the key AMFI codes
key_scheme_names = d1[d1['amfi_code'].isin(key_schemes_amfi_codes)][['amfi_code', 'scheme_name']]

# Filter d2 (NAV history) for these key schemes
key_schemes_nav = d2[d2['amfi_code'].isin(key_schemes_amfi_codes)]

# Merge to add scheme names to the NAV data
key_schemes_nav_with_names = pd.merge(key_schemes_nav, key_scheme_names, on='amfi_code', how='left')

# Convert 'date' column to datetime objects
key_schemes_nav_with_names['date'] = pd.to_datetime(key_schemes_nav_with_names['date'])

display(key_schemes_nav_with_names.head())

,amfi_code,date,nav,scheme_name
0,119551,2022-01-03,54.3856,SBI Bluechip Fund - Regular Plan - Growth
1,119551,2022-01-04,54.3474,SBI Bluechip Fund - Regular Plan - Growth
2,119551,2022-01-05,54.6869,SBI Bluechip Fund - Regular Plan - Growth
3,119551,2022-01-06,55.4550,SBI Bluechip Fund - Regular Plan - Growth
4,119551,2022-01-07,55.3692,SBI Bluechip Fund - Regular Plan - Growth


In [11]:
print("Unique Fund Houses:", d1['fund_house'].unique())

Unique Fund Houses: ['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']


In [12]:
print("\nUnique Categories:", d1['category'].unique())


Unique Categories: ['Equity' 'Debt']


In [13]:
print("\nUnique Sub-Categories:", d1['sub_category'].unique())


Unique Sub-Categories: ['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']


In [14]:
print("\nUnique Risk Categories:", d1['risk_category'].unique())


Unique Risk Categories: ['Moderate' 'Very High' 'Low' 'High' 'Moderately High']


In [15]:
amfi_codes_in_d1 = set(d1['amfi_code'].unique())
amfi_codes_in_d2 = set(d2['amfi_code'].unique())

missing_in_nav_history = amfi_codes_in_d1 - amfi_codes_in_d2

if not missing_in_nav_history:
    print("Data Quality Check: All AMFI codes from 'fund_master' are present in 'nav_history'.")
else:
    print(f"Data Quality Alert: The following AMFI codes from 'fund_master' are missing in 'nav_history': {list(missing_in_nav_history)}")
    print("This could indicate missing NAV data for these schemes.")

Data Quality Check: All AMFI codes from 'fund_master' are present in 'nav_history'.


In [18]:
#Day2